# Feature engineering - advanced data preparation pipeline  | Sebislaw

## Libraries

In [1]:
from os.path  import join
import random
import itertools
import math

import numpy as np
import pandas as pd
from pandas.plotting import scatter_matrix

import matplotlib.pyplot as plt
import plotly.express as px
from pandas.plotting import parallel_coordinates
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display

from sklearn.linear_model import LinearRegression, LassoCV, LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score, StratifiedKFold, RandomizedSearchCV
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_classif
from sklearn.neural_network import MLPClassifier

import xgboost as xgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import optuna
# from tabpfn import TabPFNClassifier

## Data

In [2]:
data_path = '..\\..\\..\\data'
pd.set_option('display.max_columns', None)

# The Basics ------------------------------------------------------------------------
# Men
MTeams = pd.read_csv(join(data_path, 'MTeams.csv'))
MSeasons = pd.read_csv(join(data_path, 'MSeasons.csv'))
MNCAATourneySeeds = pd.read_csv(join(data_path, 'MNCAATourneySeeds.csv'))
MRegularSeasonCompactResults = pd.read_csv(join(data_path, 'MRegularSeasonCompactResults.csv'))
MNCAATourneyCompactResults = pd.read_csv(join(data_path, 'MNCAATourneyCompactResults.csv'))
# Women
WTeams = pd.read_csv(join(data_path, 'WTeams.csv'))
WSeasons = pd.read_csv(join(data_path, 'WSeasons.csv'))
WNCAATourneySeeds = pd.read_csv(join(data_path, 'WNCAATourneySeeds.csv'))
WRegularSeasonCompactResults = pd.read_csv(join(data_path, 'WRegularSeasonCompactResults.csv'))
WNCAATourneyCompactResults = pd.read_csv(join(data_path, 'WNCAATourneyCompactResults.csv'))
# Other
SampleSubmissionStage1 = pd.read_csv(join(data_path, 'SampleSubmissionStage1.csv'))
SampleSubmissionStage2 = pd.read_csv(join(data_path, 'SampleSubmissionStage2.csv'))
SeedBenchmarkStage1 = pd.read_csv(join(data_path, 'SeedBenchmarkStage1.csv'))

# Team Box Scores ------------------------------------------------------------------------
# Men
MRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'MRegularSeasonDetailedResults.csv'))
MNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'MNCAATourneyDetailedResults.csv'))
# Women
WRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'WRegularSeasonDetailedResults.csv'))
WNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'WNCAATourneyDetailedResults.csv'))

# Geography ------------------------------------------------------------------------
# All
Cities = pd.read_csv(join(data_path, 'Cities.csv'))
Conferences = pd.read_csv(join(data_path, 'Conferences.csv'))
# Men
MGameCities = pd.read_csv(join(data_path, 'MGameCities.csv'))
# Women
WGameCities = pd.read_csv(join(data_path, 'WGameCities.csv'))

# Public Rankings ------------------------------------------------------------------------
# Men
MMasseyOrdinals = pd.read_csv(join(data_path, 'MMasseyOrdinals.csv')) # men only

# Supplements ------------------------------------------------------------------------
# Men
MTeamCoaches = pd.read_csv(join(data_path, 'MTeamCoaches.csv')) # men only
MTeamConferences = pd.read_csv(join(data_path, 'MTeamConferences.csv'))
MConferenceTourneyGames = pd.read_csv(join(data_path, 'MConferenceTourneyGames.csv'))
MSecondaryTourneyTeams = pd.read_csv(join(data_path, 'MSecondaryTourneyTeams.csv'))
MSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'MSecondaryTourneyCompactResults.csv'))
MTeamSpellings = pd.read_csv(join(data_path, "MTeamSpellings.csv"), encoding='cp1252')
MNCAATourneySlots = pd.read_csv(join(data_path, 'MNCAATourneySlots.csv'))
MNCAATourneySeedRoundSlots = pd.read_csv(join(data_path, 'MNCAATourneySeedRoundSlots.csv')) # men only
# Women
WTeamConferences = pd.read_csv(join(data_path, 'WTeamConferences.csv'))
WConferenceTourneyGames = pd.read_csv(join(data_path, 'WConferenceTourneyGames.csv'))
WSecondaryTourneyTeams = pd.read_csv(join(data_path, 'WSecondaryTourneyTeams.csv'))
WSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'WSecondaryTourneyCompactResults.csv'))
WTeamSpellings = pd.read_csv(join(data_path, 'WTeamSpellings.csv'), encoding='cp1252')
WNCAATourneySlots = pd.read_csv(join(data_path, 'WNCAATourneySlots.csv'))

## Data preparation pipeline

In [3]:
def prepare_data(df):
        
    """
    This function duplicates and flips a game record.
    Now two records with the same data are present, 
    but viewed from perspectives of two different teams.
    """
    
    df = df[[
         'Season', 'DayNum', 'NumOT',
         'WTeamID',  'WScore', 'WLoc',
         'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF',
         'LTeamID', 'LScore',
         'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF'
    ]]
    dfswap = df[[
         'Season', 'DayNum', 'NumOT',
         'LTeamID', 'LScore', 'WLoc',
         'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF',
         'WTeamID',  'WScore',
         'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF',
    ]].copy()
    
    dfswap.loc[df['WLoc'] == 'H', 'WLoc'] = 'A'
    dfswap.loc[df['WLoc'] == 'A', 'WLoc'] = 'H'
        
    df = df.rename(columns={'WLoc': 'location'})
    dfswap = dfswap.rename(columns={'WLoc': 'location'})
        
    df.columns = [x.replace('W','T1_').replace('L','T2_') for x in list(df.columns)]
    dfswap.columns = [x.replace('L','T1_').replace('W','T2_') for x in list(dfswap.columns)]
    
    output = pd.concat([df, dfswap]).reset_index(drop=True)
    output.loc[output.location=='N','location'] = '0'
    output.loc[output.location=='H','location'] = '1'
    output.loc[output.location=='A','location'] = '-1'
    output.location = output.location.astype(int)
        
    output['PointDiff'] = output['T1_Score'] - output['T2_Score']
    
    return output

def get_data(regular_results, tourney_results, seeds, prepared=False, location_multiplier=[1, 1], win_ratio_days_back=14):

    """
    This function uses the prepare_data function in order to create
    a data frame with season statistics for each team.
    These statistics are added to records with games played in
    tournament to make data 'x' used in model to predict the game 
    result 'y'. The output is a data frame that contains data 'x'
    and also label 'y' can be easily calculated based on score difference in matches.
    """

    if prepared:
        regular_data = regular_results.copy()
        tourney_data = tourney_results.copy()
    else:
        # make data frames with extra rows to represent the perspective of losing team
        regular_data = prepare_data(regular_results)
        tourney_data = prepare_data(tourney_results)

    # ----------------------------------- Add reward/penalty for playing in home or away
    if location_multiplier[0] == 1 and location_multiplier[1] == 1:
        # data frame with mean game statistics for a given team in a given season
        season_statistics  = regular_data.groupby(["Season", 'T1_TeamID'])[
            [
                'T1_Score', 'T1_FGM','T1_FGA','T1_FGM3','T1_FGA3','T1_FTM','T1_FTA',
                'T1_OR','T1_DR','T1_Ast','T1_TO','T1_Stl','T1_Blk','T1_PF',
                'T2_Score', 'T2_FGM','T2_FGA','T2_FGM3','T2_FGA3','T2_FTM','T2_FTA',
                'T2_OR','T2_DR','T2_Ast','T2_TO','T2_Stl','T2_Blk','T2_PF',
                'PointDiff'
            ]
        ].agg('mean').reset_index()
    else:
        # Define which columns to adjust (you can add or remove columns as needed)
        T1_cols = ['T1_Score','T1_FGM','T1_FGA','T1_FGM3','T1_FGA3','T1_FTM','T1_FTA','T1_OR','T1_DR','T1_Ast','T1_TO','T1_Stl','T1_Blk','T1_PF']
        T2_cols = ['T2_Score','T2_FGM','T2_FGA','T2_FGM3','T2_FGA3','T2_FTM','T2_FTA','T2_OR','T2_DR','T2_Ast','T2_TO','T2_Stl','T2_Blk','T2_PF']
    
        # Convert the relevant columns to float before applying the adjustment function.
        cols_to_float = T1_cols + T2_cols
        regular_data[cols_to_float] = regular_data[cols_to_float].astype(float)
        
        def adjust_stats(row):
            # Determine multipliers based on location
            if row['location'] == 1:
                factor_T1 = location_multiplier[0]  # penalize Team1 stats (home)
                factor_T2 = location_multiplier[1]  # boost Team2 stats
            elif row['location'] == -1:
                factor_T1 = location_multiplier[1]  # boost Team1 stats (away)
                factor_T2 = location_multiplier[0]  # penalize Team2 stats
            else:
                factor_T1 = 1.0
                factor_T2 = 1.0
        
            # Adjust Team1 stats
            for col in T1_cols:
                if col in row and pd.notnull(row[col]):
                    row[col] = row[col] * factor_T1
        
            # Adjust Team2 stats
            for col in T2_cols:
                if col in row and pd.notnull(row[col]):
                    row[col] = row[col] * factor_T2
        
            # Recalculate derived statistics (if needed)
            if 'T1_Score' in row and 'T2_Score' in row:
                row['PointDiff'] = row['T1_Score'] - row['T2_Score']
            return row
        
        # Apply the adjustment function row-wise.
        regular_data_adjusted = regular_data.apply(adjust_stats, axis=1)
        
        # Now group by Season and T1_TeamID to compute season averages for the adjusted statistics.
        stats_columns = T1_cols[1:] + T2_cols[1:] + ['PointDiff']  # Exclude T1_TeamID from stats if present.
        season_statistics = regular_data_adjusted.groupby(["Season", 'T1_TeamID'])[stats_columns].agg('mean').reset_index()
    # -----------------------------------
    
    # mean statistics for team and team's opponent's
    season_statistics_T1 = season_statistics.copy()
    season_statistics_T2 = season_statistics.copy()
    
    season_statistics_T1.columns = ["T1_" + x.replace("T1_","").replace("T2_","opponent_") for x in list(season_statistics_T1.columns)]
    season_statistics_T2.columns = ["T2_" + x.replace("T1_","").replace("T2_","opponent_") for x in list(season_statistics_T2.columns)]
    season_statistics_T1.columns.values[0] = "Season"
    season_statistics_T2.columns.values[0] = "Season"
    season_statistics_T1 = season_statistics_T1.rename(columns={'T1_Score': 'T1_Score_mean'})
    season_statistics_T2 = season_statistics_T2.rename(columns={'T2_Score': 'T2_Score_mean'})
    
    # data frame containing game's result
    tourney_data = tourney_data[['Season', 'DayNum', 'T1_TeamID', 'T1_Score', 'T2_TeamID' ,'T2_Score', 'location']]
    tourney_data = pd.merge(tourney_data, season_statistics_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, season_statistics_T2, on = ['Season', 'T2_TeamID'], how = 'left')

    calculate_win_ratio_days_back = 132 - win_ratio_days_back
    
    # data frame with win fraction from last x days for a given team in a given season
    last14days_stats_T1 = regular_data.loc[regular_data.DayNum>calculate_win_ratio_days_back].reset_index(drop=True)
    last14days_stats_T1['win'] = np.where(last14days_stats_T1['PointDiff']>0,1,0)
    last14days_stats_T1 = last14days_stats_T1.groupby(['Season','T1_TeamID'])['win'].mean().reset_index(name='T1_win_ratio_14d')
    
    last14days_stats_T2 = regular_data.loc[regular_data.DayNum>calculate_win_ratio_days_back].reset_index(drop=True)
    last14days_stats_T2['win'] = np.where(last14days_stats_T2['PointDiff']<0,1,0)
    last14days_stats_T2 = last14days_stats_T2.groupby(['Season','T2_TeamID'])['win'].mean().reset_index(name='T2_win_ratio_14d')
    
    # add to tourney_data column with win fraction for winning and losing team
    tourney_data = pd.merge(tourney_data, last14days_stats_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, last14days_stats_T2, on = ['Season', 'T2_TeamID'], how = 'left')
    
    # get seeds with no regional division
    seeds['seed'] = seeds['Seed'].apply(lambda x: int(x[1:3]))
    
    # give each team a raw seed
    seeds_T1 = seeds[['Season','TeamID','seed']].copy()
    seeds_T2 = seeds[['Season','TeamID','seed']].copy()
    seeds_T1.columns = ['Season','T1_TeamID','T1_seed']
    seeds_T2.columns = ['Season','T2_TeamID','T2_seed']
    
    # add seeds to turney data for team 1 and team 2
    tourney_data = pd.merge(tourney_data, seeds_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, seeds_T2, on = ['Season', 'T2_TeamID'], how = 'left')
    
    # add a seed difference column
    tourney_data["Seed_diff"] = tourney_data["T1_seed"] - tourney_data["T2_seed"]

    return tourney_data

def get_df(seeds,
            season_games, season_range, days_back,
            tourney_games, tourney_range, 
            location_multiplier=[1, 1],
           win_ratio_days_back=14):

    """
    This function uses get_data function to get a data
    frame which is then used to make 'x' and 'y' data
    used in models.
    """
    
    # Get from regular season games from specified season range and days back
    # The results of those games will be used as team statictics (additional team information for tourney games)
    regular_results = season_games[season_games['Season'].isin(season_range)]
    regular_results = regular_results[regular_results['DayNum'] > 134 - days_back]
    
    # Get tourney games from a specified season
    # The results of those games will be used as labels
    tourney_results = tourney_games[tourney_games['Season'].isin(tourney_range)]
    
    # Get final data frame
    df = get_data(regular_results, tourney_results, seeds, location_multiplier=[1, 1], win_ratio_days_back=win_ratio_days_back)
    
    return df

def get_final_df(seeds,
                   season_games, season_range, days_back,
                   tourney_games, tourney_range, 
                   SampleSubmissionStage1,
                  location_multiplier=[1, 1],
                win_ratio_days_back=14):

    """
    This function works the same as function get_x_y,
    but also creates (at the moment it's the same as get_x_y)
    aditional data points mostly with NaN values that matches
    the submission format (parsed team matchups with the sample submission file).
    """

    # Get from regular season games from specified season range and days back
    # The results of those games will be used as team statictics (additional team information for tourney games)
    regular_results = season_games[season_games['Season'].isin(season_range)]
    regular_results = regular_results[regular_results['DayNum'] > 134 - days_back]

    # Get tourney games from a specified season
    # The results of those games will be used as labels
    tourney_results = tourney_games[tourney_games['Season'].isin(tourney_range)]
    
    # Assume sample_submission is a DataFrame with an "ID" column like "2023_1101_1102"
    # and tourney_results is a DataFrame with columns including: Season, WTeamID, LTeamID, DayNum, WScore, LScore, WLoc, etc.
    # Filter rows where the ID starts with the specified season (followed by an underscore)
    final_season = tourney_range[0]
    sample_submission_copy = SampleSubmissionStage1.copy()
    sample_submission = sample_submission_copy[sample_submission_copy['ID'].str.startswith(f"{final_season}_")]
    sample_submission = sample_submission.drop(columns=['Pred'])
    
    sample_submission[['Season', 'Team1', 'Team2']] = sample_submission['ID'].str.split('_', expand=True)
    sample_submission['Season'] = sample_submission['Season'].astype(float)
    sample_submission['Team1'] = sample_submission['Team1'].astype(float)
    sample_submission['Team2'] = sample_submission['Team2'].astype(float)
    
    regular_data_final = prepare_data(regular_results)
    tourney_data_final  = prepare_data(tourney_results)
    
    tourney_data =  get_data(regular_data_final, tourney_data_final,
                             seeds, prepared=True,
                          win_ratio_days_back=win_ratio_days_back)
    tourney_data = pd.merge(
        sample_submission,
        tourney_data,
        left_on=['Season', 'Team1', 'Team2'],
        right_on=['Season', 'T1_TeamID', 'T2_TeamID'],
        how='left'
    )
    tourney_data['T1_TeamID'] = tourney_data['Team1']
    tourney_data['T2_TeamID'] = tourney_data['Team2']
    tourney_data = tourney_data.drop(['ID', 'Team1', 'Team2'], axis=1)
    
    return tourney_data

def correct_predictions_based_on_seed(x, y, maximum_favoured_seed = 4, number_of_added_columns=1):
    """
    Sets the winnning chance to 1 or 0 based on the seed difference.
    """
    for i in range(len(x)):
        if x[i][-1-number_of_added_columns] <= (-16 + maximum_favoured_seed * 2 - 1):
            y[i] = 1
        elif x[i][-1-number_of_added_columns] >= (16 - maximum_favoured_seed * 2 + 1):
            y[i] = 0
    return y

def clear_na_from_x_y(x, y):
    """
    The data frame for final season is in format matching the submission file.
    This function clears NaNs from data.
    """
    # Create masks for training data:
    mask_train = ~np.isnan(x).any(axis=1) & ~np.isnan(y)
    x_clean = x[mask_train]
    y_clean = y[mask_train]
    return x_clean, y_clean

def x_y_from_data_frame(df):
    # Prepare data and labels
    x = df[list(df.columns[7:])].values
    y = np.where(
        df[['T1_Score', 'T2_Score']].isnull().any(axis=1),
        np.nan,
        np.where(df['T1_Score'] - df['T2_Score'] > 0, 1, 0)
    )
    return x, y

def get_all_core_data(
    regular_results, tourney_results, seeds, SampleSubmissionStage1,
    final_season = 2024, # the season we want to predict, so for out submission it will be 2025
    start_season = 2005, # from which ponit should we begin creating data
    season_years_list  = [[i-1, i] for i in range(2005, 2024+1)], # at which seasons to look at when calculating team's stats
    days_back = 15, # how many days back from the start of tourney to calculate team's stats per season
    maximum_favoured_seed = 4, # Set the predicted probability of winning to 1 for seeds <= maximum_favoured_seed and to 0 for >= 16-maximum_favoured_seed
    location_multiplier=[0.95, 1.05], # home penalty, away bonus 
    include_men = True, # include M... data sets when preparing x and y
    include_women = True, # include W... data sets when preparing x and y
    win_ratio_days_back = 14):

    """
    The idea of this function is to easily get data needed to train and test the model later on, with minimal code
    to not clutter the netebook.
    
    This function outputs df, x, y, df_final, x_final_season, y_final_season.
    
    df is a data frame with first 6 columns from tourney games and other calculated from other data frames.
    
    Adding a column to df and executing x_y_from_data_frame(df) function will yield x with added data.
    
    Note that df_final has the same structure as df, but also with rows with NaNs. The rows with missing information are there
    to match the sumbission file format. The separation of those data frames is to ensure that information from last season doesn't
    leak into training data due to poorly written code.
    
    x has df columns from location onwardsthe columns before that are from tourney games and are used to calculate y (based on points).
    
    y has label 0 or 1 (lose or win) and nan if the correspoinding data in x was nan.
    
    There is also x_final_season and y_final_season aquired from df_final, which are the same as x and y, but like df_final, they have NaNs. 
    """

    # This ensures that we simulate the scenario in competition
    tourney_results_final = tourney_results[tourney_results['Season'] == final_season]
    tourney_results = tourney_results[tourney_results['Season'] < final_season]
    regular_results = regular_results[regular_results['Season'] != 2020] # This year had no tournament data
    tourney_years_list = [[i] for i in range(start_season, final_season+1)]
    
    # Arrays to store data
    data = []
    data_final = []
    for season_years, tourney_years in zip(season_years_list, tourney_years_list):
            
        # Create data separately for the of games
        # The separation is to ensure there is no data leak
        if tourney_years[0] == final_season:
            data_tmp = get_df(
                seeds,
                regular_results, season_years, days_back,
                tourney_results_final, tourney_years,
                location_multiplier=location_multiplier,
                win_ratio_days_back=win_ratio_days_back
            )
            data_final.append(data_tmp)
        else:
            data_tmp = get_df(
                seeds,
                regular_results, season_years, days_back,
                tourney_results, tourney_years,
                location_multiplier=location_multiplier,
                win_ratio_days_back=win_ratio_days_back
            )
            data.append(data_tmp)
            
    df = pd.concat(data, ignore_index=True)
    df_final = pd.concat(data_final, ignore_index=True)
    
    return df, df_final

def brier_for_all_years(year_range, season_years_list):
    
    df_train_women_list = []
    df_test_women_list = []
    df_train_men_list = []
    df_test_men_list = []
    
    for year, season_years in zip(year_range, season_years_list):
    
        final_season = year

        for sex in ['woman', 'man']:
            
            if sex == 'woman':
                include_men = False
                include_women = True
            elif sex == 'man':
                include_men = True
                include_women = False

            # ----------------------------------------------------------
            # READ DATA
            regular_results = pd.concat([
                MRegularSeasonDetailedResults.copy() if include_men else None,
                WRegularSeasonDetailedResults.copy() if include_women else None
            ], ignore_index=True)
            tourney_results = pd.concat([
                MNCAATourneyDetailedResults.copy() if include_men else None,
                WNCAATourneyDetailedResults.copy() if include_women else None
            ], ignore_index=True)
            seeds = pd.concat([
                MNCAATourneySeeds.copy() if include_men else None,
                WNCAATourneySeeds.copy() if include_women else None
            ], ignore_index=True)
            # ----------------------------------------------------------
            # GET ALL DATA NEEDED TO USE THE MODELS
            df_train, df_test = get_all_core_data(
                regular_results, tourney_results, seeds, SampleSubmissionStage1, final_season = final_season,
                start_season = start_season, season_years_list  = season_years, days_back = days_back,
                maximum_favoured_seed = maximum_favoured_seed, location_multiplier=location_multiplier,
                include_men = include_men, include_women = include_women, win_ratio_days_back = win_ratio_days_back)
            # ----------------------------------------------------------
            # ADD TEAM AND COACH ELO AND REPLACE NAN WITH MEAN
            elo = pd.read_csv(join(data_path, 'elo.csv'))
            elo['CoachELO'] = elo['CoachELO'].fillna(elo['CoachELO'].mean())
            elo = elo.drop(['CoachName'], axis=1)
            def add_elo_column(df):
                df = df.copy()
                df = pd.merge(
                        df,
                        elo[['Season', 'DayNum', 'TeamID', 'TeamELO', 'CoachELO']],
                        left_on=['Season', 'DayNum', 'T1_TeamID'],
                        right_on=['Season', 'DayNum', 'TeamID'],
                        how='left'
                    )
                df = df.drop(['TeamID'], axis=1)
                return df
            df_train = add_elo_column(df_train)
            df_test = add_elo_column(df_test)
            # ----------------------------------------------------------
            if sex == 'woman':
                df_train_women_list.append(df_train.copy())
                df_test_women_list.append(df_test.copy())
            elif sex == 'man':
                df_train_men_list.append(df_train.copy())
                df_test_men_list.append(df_test.copy())
                
        for i in range(len(df_train_men_list)):
            df_train_women_list[i] = df_train_women_list[i][list(df_train_men_list[0])]
            df_test_women_list[i] = df_test_women_list[i][list(df_train_men_list[0])]
            
    return df_train_women_list, df_test_women_list, df_train_men_list, df_test_men_list

## Get data to use models on

In [4]:
columns_to_include_women = [
 'Season',
 'DayNum',
 'T1_TeamID',
 'T1_Score',
 'T2_TeamID',
 'T2_Score',
 'location',
                      
#  'T1_Score_mean',
 'T1_FGM',
 'T1_FGA',
 'T1_FGM3',
 'T1_FGA3',
#  'T1_FTM',
#  'T1_FTA',
 'T1_OR',
#  'T1_DR',
 'T1_Ast',
 'T1_TO',
 'T1_Stl',
#  'T1_Blk',
 'T1_PF',
                      
#  'T1_opponent_Score',
 'T1_opponent_FGM',
 'T1_opponent_FGA',
 'T1_opponent_FGM3',
 'T1_opponent_FGA3',
#  'T1_opponent_FTM',
#  'T1_opponent_FTA',
 'T1_opponent_OR',
#  'T1_opponent_DR',
 'T1_opponent_Ast',
 'T1_opponent_TO',
 'T1_opponent_Stl',
#  'T1_opponent_Blk',
 'T1_opponent_PF',
 'T1_PointDiff',
                      
#  'T2_Score_mean',
 'T2_FGM',
 'T2_FGA',
 'T2_FGM3',
 'T2_FGA3',
#  'T2_FTM',
#  'T2_FTA',
 'T2_OR',
#  'T2_DR',
 'T2_Ast',
 'T2_TO',
 'T2_Stl',
#  'T2_Blk',
 'T2_PF',
                      
#  'T2_opponent_Score',
 'T2_opponent_FGM',
 'T2_opponent_FGA',
 'T2_opponent_FGM3',
 'T2_opponent_FGA3',
#  'T2_opponent_FTM',
#  'T2_opponent_FTA',
 'T2_opponent_OR',
#  'T2_opponent_DR',
 'T2_opponent_Ast',
 'T2_opponent_TO',
 'T2_opponent_Stl',
#  'T2_opponent_Blk',
 'T2_opponent_PF',
 'T2_PointDiff',
                      
 'T1_win_ratio_14d',
 'T2_win_ratio_14d',
 'T1_seed',
 'T2_seed',
 'Seed_diff',
 'TeamELO',
 'CoachELO'
]
columns_to_include_men = [
 'Season',
 'DayNum',
 'T1_TeamID',
 'T1_Score',
 'T2_TeamID',
 'T2_Score',
 'location',
                      
#  'T1_Score_mean',
 'T1_FGM',
 'T1_FGA',
 'T1_FGM3',
 'T1_FGA3',
#  'T1_FTM',
#  'T1_FTA',
 'T1_OR',
#  'T1_DR',
 'T1_Ast',
 'T1_TO',
 'T1_Stl',
#  'T1_Blk',
 'T1_PF',
                      
#  'T1_opponent_Score',
 'T1_opponent_FGM',
 'T1_opponent_FGA',
 'T1_opponent_FGM3',
 'T1_opponent_FGA3',
#  'T1_opponent_FTM',
#  'T1_opponent_FTA',
 'T1_opponent_OR',
#  'T1_opponent_DR',
 'T1_opponent_Ast',
 'T1_opponent_TO',
 'T1_opponent_Stl',
#  'T1_opponent_Blk',
 'T1_opponent_PF',
 'T1_PointDiff',
                      
#  'T2_Score_mean',
 'T2_FGM',
 'T2_FGA',
 'T2_FGM3',
 'T2_FGA3',
#  'T2_FTM',
#  'T2_FTA',
 'T2_OR',
#  'T2_DR',
 'T2_Ast',
 'T2_TO',
 'T2_Stl',
#  'T2_Blk',
 'T2_PF',
                      
#  'T2_opponent_Score',
 'T2_opponent_FGM',
 'T2_opponent_FGA',
 'T2_opponent_FGM3',
 'T2_opponent_FGA3',
#  'T2_opponent_FTM',
#  'T2_opponent_FTA',
 'T2_opponent_OR',
#  'T2_opponent_DR',
 'T2_opponent_Ast',
 'T2_opponent_TO',
 'T2_opponent_Stl',
#  'T2_opponent_Blk',
 'T2_opponent_PF',
 'T2_PointDiff',
                      
 'T1_win_ratio_14d',
 'T2_win_ratio_14d',
 'T1_seed',
 'T2_seed',
 'Seed_diff',
 'TeamELO',
 'CoachELO'
]

year_range = [2022]
start_season = 2010 # from which ponit should we begin creating data
season_years_list  = [[[i] for i in range(start_season, year+1)] for year in year_range] # at which seasons to look at when calculating team's stats
days_back = 25 # how many days back from the start of tourney to calculate team's stats per season
location_multiplier = [0.95, 1.05] # home penalty, away bonus ex.
win_ratio_days_back = 14 # how many days back from the tourney do we calculate win ratio
maximum_favoured_seed = 0
# -------------------------------------------
df_train_women_list, df_test_women_list, df_train_men_list, df_test_men_list = brier_for_all_years(year_range, season_years_list)
x_train_women_list = []
x_test_women_list = []
x_train_men_list = []
x_test_men_list = []
y_train_women_list = []
y_test_women_list = []
y_train_men_list = []
y_test_men_list = []
for i in range(len(year_range)):

    if len(year_range) > 1:
        df_train_women_list[i] = df_train_women_list[i][columns_to_include_women]
        df_test_women_list[i] = df_test_women_list[i][columns_to_include_women]
        df_train_men_list[i] = df_train_men_list[i][columns_to_include_men]
        df_test_men_list[i] = df_test_men_list[i][columns_to_include_men]
        
        x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list[i])
        x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list[i])
        x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list[i])
        x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list[i])

        x_train_women, y_train_women = clear_na_from_x_y(x_train_women.copy(), y_train_women.copy())
        x_test_women, y_test_women = clear_na_from_x_y(x_test_women.copy(), y_test_women.copy())
        x_train_men, y_train_men = clear_na_from_x_y(x_train_men.copy(), y_train_men.copy())
        x_test_men, y_test_men = clear_na_from_x_y(x_test_men.copy(), y_test_men.copy())
        
        x_train_women_list.append(x_train_women)
        x_test_women_list.append(x_test_women)
        x_train_men_list.append(x_train_men)
        x_test_men_list.append(x_test_men)

        y_train_women_list.append(y_train_women)
        y_test_women_list.append(y_test_women)
        y_train_men_list.append(y_train_men)
        y_test_men_list.append(y_test_men)
    
    else:
        df_train_women_list = df_train_women_list[0][columns_to_include_women]
        df_test_women_list = df_test_women_list[0][columns_to_include_women]
        df_train_men_list = df_train_men_list[0][columns_to_include_men]
        df_test_men_list = df_test_men_list[0][columns_to_include_men]

        x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list)
        x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list)
        x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list)
        x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list)

        x_train_women, y_train_women = clear_na_from_x_y(x_train_women.copy(), y_train_women.copy())
        x_test_women, y_test_women = clear_na_from_x_y(x_test_women.copy(), y_test_women.copy())
        x_train_men, y_train_men = clear_na_from_x_y(x_train_men.copy(), y_train_men.copy())
        x_test_men, y_test_men = clear_na_from_x_y(x_test_men.copy(), y_test_men.copy())

# Training models

## Logistic regression

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import optuna
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import brier_score_loss

# Convert NumPy arrays to PyTorch tensors
def to_tensor(X, y):
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32).unsqueeze(1)

# Define the neural network model
class NeuralNet(nn.Module):
    def __init__(self, input_size, hidden_sizes, dropout_rate):
        super(NeuralNet, self).__init__()
        layers = []
        prev_size = input_size

        for size in hidden_sizes:
            layers.append(nn.Linear(prev_size, size))
            layers.append(nn.BatchNorm1d(size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_size = size

        layers.append(nn.Linear(prev_size, 1))  # Output layer
        layers.append(nn.Sigmoid())  # Sigmoid for probability output

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

# Define the training function
def train_nn(model, X_train, y_train, X_val, y_val, learning_rate, epochs=100, batch_size=32):
    criterion = nn.BCELoss()  # Binary Cross-Entropy Loss
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    train_loader = torch.utils.data.DataLoader(
        list(zip(X_train, y_train)), batch_size=batch_size, shuffle=True
    )

    val_loader = torch.utils.data.DataLoader(
        list(zip(X_val, y_val)), batch_size=batch_size, shuffle=False
    )

    best_val_loss = float("inf")
    early_stopping_counter = 0

    for epoch in range(epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

        # Validation
        model.eval()
        with torch.no_grad():
            val_losses = []
            for X_batch, y_batch in val_loader:
                outputs = model(X_batch)
                val_losses.append(criterion(outputs, y_batch).item())

            avg_val_loss = np.mean(val_losses)

            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                early_stopping_counter = 0  # Reset early stopping counter
            else:
                early_stopping_counter += 1
                if early_stopping_counter >= 5:
                    break  # Stop training if validation loss does not improve

    return model

# Define the objective function for Optuna
def objective_nn(trial, X, y, n_folds=5):
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5)
    hidden_layer_1 = trial.suggest_int("hidden_1", 32, 256, step=32)
    hidden_layer_2 = trial.suggest_int("hidden_2", 32, 256, step=32)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in skf.split(X, y):
        X_train_cv, X_val_cv = X[train_idx], X[val_idx]
        y_train_cv, y_val_cv = y[train_idx], y[val_idx]

        X_train_cv, y_train_cv = to_tensor(X_train_cv, y_train_cv)
        X_val_cv, y_val_cv = to_tensor(X_val_cv, y_val_cv)

        model = NeuralNet(input_size=X.shape[1], hidden_sizes=[hidden_layer_1, hidden_layer_2], dropout_rate=dropout_rate)
        model = train_nn(model, X_train_cv, y_train_cv, X_val_cv, y_val_cv, learning_rate)

        model.eval()
        with torch.no_grad():
            y_pred = model(X_val_cv).numpy().flatten()

        scores.append(brier_score_loss(y_val_cv.numpy(), y_pred))

    return np.mean(scores)

# Train the best neural network for each dataset
def train_best_nn(X_train, y_train, X_test, y_test):
    study = optuna.create_study(direction="minimize")
    study.optimize(lambda trial: objective_nn(trial, X_train, y_train), n_trials=50)

    best_params = study.best_params
    print("Best Neural Network Params:", best_params)

    X_train_torch, y_train_torch = to_tensor(X_train, y_train)
    X_test_torch, y_test_torch = to_tensor(X_test, y_test)

    best_nn = NeuralNet(input_size=X_train.shape[1],
                        hidden_sizes=[best_params["hidden_1"], best_params["hidden_2"]],
                        dropout_rate=best_params["dropout_rate"])

    best_nn = train_nn(best_nn, X_train_torch, y_train_torch, X_test_torch, y_test_torch, best_params["learning_rate"])

    # Predict on test set
    best_nn.eval()
    with torch.no_grad():
        y_pred = best_nn(X_test_torch).numpy().flatten()

    # Compute Brier Score
    brier_score = brier_score_loss(y_test, y_pred)
    print("Final Neural Network Brier Score:", brier_score)

    return best_nn, y_pred

# Train for Women
print("Training Neural Network for Women...")
nn_women, y_pred_women_nn = train_best_nn(x_train_women, y_train_women, x_test_women, y_test_women)

# Train for Men
print("Training Neural Network for Men...")
nn_men, y_pred_men_nn = train_best_nn(x_train_men, y_train_men, x_test_men, y_test_men)


C:\Users\Sebastian\anaconda3\lib\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(
[I 2025-03-20 00:06:43,012] A new study created in memory with name: no-name-cd0ade18-265f-4c8f-8d57-a53da9b412ea


Training Neural Network for Women...


[I 2025-03-20 00:06:57,406] Trial 0 finished with value: 0.2600594444359955 and parameters: {'dropout_rate': 0.1587410225355977, 'hidden_1': 96, 'hidden_2': 256, 'learning_rate': 0.004602683697063052}. Best is trial 0 with value: 0.2600594444359955.
[I 2025-03-20 00:07:09,080] Trial 1 finished with value: 0.18513980723884027 and parameters: {'dropout_rate': 0.43277967788441496, 'hidden_1': 32, 'hidden_2': 96, 'learning_rate': 0.0018605221611008981}. Best is trial 1 with value: 0.18513980723884027.
[I 2025-03-20 00:07:17,994] Trial 2 finished with value: 0.26274014886469577 and parameters: {'dropout_rate': 0.11931019177729084, 'hidden_1': 224, 'hidden_2': 96, 'learning_rate': 0.0022164634693033697}. Best is trial 1 with value: 0.18513980723884027.
[I 2025-03-20 00:07:27,852] Trial 3 finished with value: 0.20600772998107372 and parameters: {'dropout_rate': 0.2392880331137869, 'hidden_1': 32, 'hidden_2': 256, 'learning_rate': 0.002420185123034734}. Best is trial 1 with value: 0.1851398072

[I 2025-03-20 00:19:19,687] Trial 33 finished with value: 0.14675121866054852 and parameters: {'dropout_rate': 0.4740257365291407, 'hidden_1': 96, 'hidden_2': 96, 'learning_rate': 0.00013985651180008213}. Best is trial 27 with value: 0.14652699079769255.
[I 2025-03-20 00:19:31,938] Trial 34 finished with value: 0.15720994878771033 and parameters: {'dropout_rate': 0.10013408403567425, 'hidden_1': 128, 'hidden_2': 96, 'learning_rate': 0.00026974093180118247}. Best is trial 27 with value: 0.14652699079769255.
[I 2025-03-20 00:20:27,720] Trial 35 finished with value: 0.15029000436786188 and parameters: {'dropout_rate': 0.4495923114231925, 'hidden_1': 32, 'hidden_2': 96, 'learning_rate': 0.0001021136674858514}. Best is trial 27 with value: 0.14652699079769255.
[I 2025-03-20 00:21:07,181] Trial 36 finished with value: 0.14617273158690167 and parameters: {'dropout_rate': 0.37002005712283914, 'hidden_1': 96, 'hidden_2': 96, 'learning_rate': 0.00013781444759918403}. Best is trial 36 with value:

Best Neural Network Params: {'dropout_rate': 0.37002005712283914, 'hidden_1': 96, 'hidden_2': 96, 'learning_rate': 0.00013781444759918403}


[I 2025-03-20 00:28:15,444] A new study created in memory with name: no-name-9b8301c5-4dda-4094-9051-a5c6937bdbc6


Final Neural Network Brier Score: 0.17345813891617193
Training Neural Network for Men...


[I 2025-03-20 00:28:29,598] Trial 0 finished with value: 0.2708739993882196 and parameters: {'dropout_rate': 0.3935947304844225, 'hidden_1': 32, 'hidden_2': 96, 'learning_rate': 0.009118350747043097}. Best is trial 0 with value: 0.2708739993882196.
[I 2025-03-20 00:28:40,895] Trial 1 finished with value: 0.2327646690121739 and parameters: {'dropout_rate': 0.25182592212370913, 'hidden_1': 256, 'hidden_2': 160, 'learning_rate': 0.0007742047739656058}. Best is trial 1 with value: 0.2327646690121739.
[I 2025-03-20 00:28:52,682] Trial 2 finished with value: 0.23260784497709505 and parameters: {'dropout_rate': 0.2964468977898056, 'hidden_1': 96, 'hidden_2': 32, 'learning_rate': 0.0025351268505033483}. Best is trial 2 with value: 0.23260784497709505.
[I 2025-03-20 00:29:23,801] Trial 3 finished with value: 0.2086902591795316 and parameters: {'dropout_rate': 0.3416943934615103, 'hidden_1': 64, 'hidden_2': 160, 'learning_rate': 0.00015857665792069368}. Best is trial 3 with value: 0.208690259179

[I 2025-03-20 00:38:10,594] Trial 33 finished with value: 0.21191921362240906 and parameters: {'dropout_rate': 0.2624348244166589, 'hidden_1': 128, 'hidden_2': 96, 'learning_rate': 0.00020384968031347657}. Best is trial 10 with value: 0.20496311935769135.
[I 2025-03-20 00:38:41,326] Trial 34 finished with value: 0.20718725627382573 and parameters: {'dropout_rate': 0.37645529777818876, 'hidden_1': 64, 'hidden_2': 128, 'learning_rate': 0.00012898880380524122}. Best is trial 10 with value: 0.20496311935769135.
[I 2025-03-20 00:39:07,023] Trial 35 finished with value: 0.21053561710215024 and parameters: {'dropout_rate': 0.45181259681670405, 'hidden_1': 256, 'hidden_2': 128, 'learning_rate': 0.00018694149802299937}. Best is trial 10 with value: 0.20496311935769135.
[I 2025-03-20 00:39:42,120] Trial 36 finished with value: 0.2075805390714122 and parameters: {'dropout_rate': 0.3838557776098932, 'hidden_1': 96, 'hidden_2': 64, 'learning_rate': 0.00012456715476317107}. Best is trial 10 with val

Best Neural Network Params: {'dropout_rate': 0.42795302733969826, 'hidden_1': 96, 'hidden_2': 64, 'learning_rate': 0.0002408868709275173}
Final Neural Network Brier Score: 0.20766445980364637
